# Imports

In [13]:
import os

import pandas as pd
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    DateType,
    FloatType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)
from rich import print

# Data

- Get Weather Data from Cincinnati & Florida from [NCEI.nooa.gov](https://www.ncei.noaa.gov/data/global-summary-of-the-day/access/)
- Required data:
    - Cincinnati (72429793812) from 2015 - 2024
    - Florida (99495199999) from 2015 - 2024 (note that 2016 Florida data does not exist)

In [ ]:
# script to download the required data
def download_weather_data(city_code, start_year, end_year, output_dir):
    """
    Downloads weather data for a given city and time range, and saves it to the specified output directory.

    Args:
        city_code (str): The code of the city for which to download weather data.
        start_year (int): The starting year of the data to download.
        end_year (int): The ending year of the data to download.
        output_dir (str): The directory where the downloaded data should be saved.
    
    Returns:
        None
    """


    # make sure output directory exists
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    for year in range(start_year, end_year + 1):
        # Construct the URL of ncei.noaa.gov for the specified city and time range
        url = f"https://www.ncei.noaa.gov/data/global-summary-of-the-day/access/{year}/{city_code}.csv"
        # Download the data and save it to the output directory
        sub_dir = os.path.join(output_dir, str(year))
        if not os.path.exists(sub_dir):
            os.makedirs(sub_dir)
        output_file = os.path.join(sub_dir, f"{city_code}.csv")
        try:
            # os.system(f"curl -o {output_file} {url}")   
            # The -f flag tells curl to return an error code on 404
            exit_code = os.system(f"curl -f -o {output_file} {url} --silent")   
            
            if exit_code == 0:
                print(f"[green](*)[/green] Data for {city_code} in {year} has been downloaded and saved to {output_file}.")
            else:
                print(f"[red](!)[/red] Failed to download data for {city_code} in {year}. (Likely 404 Not Found)")
                # Optional: Remove the empty file curl might have created before failing
                if os.path.exists(output_file):
                    os.remove(output_file)
            # print(f"(*) Data for {city_code} in {year} has been downloaded and saved to {output_file}.")
        except Exception as e:
            print(f"[red](!)[/red] An error occurred while downloading data for {city_code} in {year}: {e}")

In [15]:
# run the script to download the data for the specified city and time range
CINCINNATI_WEATHER_CODE = "72429793812"
FLORIDA_WEATHER_CODE = "99495199999"
START_YEAR = 2015
END_YEAR = 2024

download_weather_data(CINCINNATI_WEATHER_CODE, START_YEAR, END_YEAR, "data/")
download_weather_data(FLORIDA_WEATHER_CODE, START_YEAR, END_YEAR, "data/")

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0curl: (6) Could not resolve host: data
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0

"STATION","DATE","LATITUDE","LONGITUDE","ELEVATION","NAME","TEMP","TEMP_ATTRIBUTES","DEWP","DEWP_ATTRIBUTES","SLP","SLP_ATTRIBUTES","STP","STP_ATTRIBUTES","VISIB","VISIB_ATTRIBUTES","WDSP","WDSP_ATTRIBUTES","MXSPD","GUST","MAX","MAX_ATTRIBUTES","MIN","MIN_ATTRIBUTES","PRCP","PRCP_ATTRIBUTES","SNDP","FRSHTT"
"72429793812","2015-01-01","39.106","-84.41609","144.8","CINCINNATI MUNICIPAL AIRPORT LUNKEN FIELD, OH US","  26.1","24","  10.9","24","1025.5","24","006.8","24"," 10.0","24"," 10.0","24"," 18.1"," 22.9","  39.0"," ","  15.1"," "," 0.00","G","999.9","000000"
"72429793812","2015-01-02","39.106","-84.41609","144.8","CINCINNATI MUNICIPAL AIRPORT LUNKEN FIELD, OH US","  33.3","24","  25.3","24","1026.4","22","007.9","24"," 10.0","24","  4.1","24","  8.9","999.9","  39.0"," ","  18.0"," "," 0.00","G","999.9","000000"
"72429793812","2015-01-03","39.106","-84.41609","144.8","CINCINNATI MUNICIPAL AIRPORT LUNKEN FIELD, OH US","  41.8","24","  39.9","24","1020.4","13","000.7","24","  6.1","24

100 94844  100 94844    0     0   179k      0 --:--:-- --:--:-- --:--:--  179k


(*) Data for 72429793812 in 2015 has been downloaded and saved to data/2015/72429793812.csv.

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0curl: (6) Could not resolve host: data
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0

...

(*) Data for 99495199999 in 2023 has been downloaded and saved to data/2023/99495199999.csv.

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0curl: (6) Could not resolve host: data
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0

"STATION","DATE","LATITUDE","LONGITUDE","ELEVATION","NAME","TEMP","TEMP_ATTRIBUTES","DEWP","DEWP_ATTRIBUTES","SLP","SLP_ATTRIBUTES","STP","STP_ATTRIBUTES","VISIB","VISIB_ATTRIBUTES","WDSP","WDSP_ATTRIBUTES","MXSPD","GUST","MAX","MAX_ATTRIBUTES","MIN","MIN_ATTRIBUTES","PRCP","PRCP_ATTRIBUTES","SNDP","FRSHTT"
"99495199999","2024-01-03","27.862","-80.445","10.0","SEBASTIAN INLET STATE PARK, FL US","  59.1","24","9999.9"," 0","1018.0","24","999.9"," 0","999.9"," 0","  8.2","24"," 14.0","999.9","  68.4","*","  53.2","*"," 0.00","I","999.9","000000"
"99495199999","2024-01-04","27.862","-80.445","10.0","SEBASTIAN INLET STATE PARK, FL US","  62.6","22","9999.9"," 0","1014.2","22","999.9"," 0","999.9"," 0"," 10.9","22"," 22.0","999.9","  65.5","*","  59.2","*"," 0.00","I","999.9","000000"
"99495199999","2024-01-05","27.862","-80.445","10.0","SEBASTIAN INLET STATE PARK, FL US","  67.2","24","9999.9"," 0","1018.1","24","999.9"," 0","999.9"," 0"," 15.2","24"," 21.0","999.9","  69.8","*","  61.3","

100 32362  100 32362    0     0   154k      0 --:--:-- --:--:-- --:--:--  154k


(*) Data for 99495199999 in 2024 has been downloaded and saved to data/2024/99495199999.csv.